# Paper figures

All numbers come from `paper_data.py` and are cached in `data/paper_cache/` (set `REBUILD = True`, or delete a cache file, to recompute one).
Conventions are listed in `paper_data.py`'s docstring: (A+B)/2 lattice spectra; wall_points=100 pair-spectrum scans at each run's exact
$\bar\lambda$; our BubbleMaster cutoff window; $\hat D^{3.6}$ damping; lattice-vs-surrogate comparisons through the lattice shell estimator;
$I_{\mathcal K}=\sum_n P_n\,\Delta k/k_n$ over shells with $kR_*\le100$. Figures go to `notebooks/plots/paper/`.

In [ ]:
#%pip install -e ../.

In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
from matplotlib import rc, rcParams
from matplotlib.colors import LogNorm
from matplotlib.ticker import FixedLocator, NullLocator

import paper_data as P

# ---------------------------------------------------------------- global settings
REBUILD = False                  # True: recompute every cache (slow); or delete one file in data/paper_cache/

TEXTWIDTH = 7.0                  # inches; figure spans both columns (check with \the\textwidth in the paper)
FONTSIZE = 9                     # pt, all text in every figure (caption size)
LEGEND_FONTSIZE = 8              # pt, legends (plain text reads larger than the maths labels)

LW = 1.0                         # main line width
LW_THIN = 0.75                   # secondary lines (ratios, dashed surrogates)
LS_UNDAMPED = ':'               # undamped surrogate (dotted is taken by the ratio = 1 line)

OUT = P.ROOT / 'notebooks' / 'plots' / 'paper'
OUT.mkdir(parents=True, exist_ok=True)

# on-screen display only (the saved PDFs are vector graphics at the print size above)
%config InlineBackend.figure_format = 'retina'
DISPLAY_DPI = 150                # larger number = larger figure in the notebook
rcParams['figure.dpi'] = DISPLAY_DPI
rcParams['savefig.dpi'] = 300    # resolution of the rasterized layers (heatmaps) inside the PDFs

rc('font', **{'family': 'serif', 'serif': ['Computer Modern']})
rcParams['text.usetex'] = True
rcParams.update({
    'font.size': FONTSIZE,
    'axes.labelsize': FONTSIZE,
    'axes.titlesize': FONTSIZE,
    'legend.fontsize': LEGEND_FONTSIZE,
    'xtick.labelsize': FONTSIZE,
    'ytick.labelsize': FONTSIZE,
    'legend.handlelength': 1.2,
    'legend.handletextpad': 0.5,
    'legend.labelspacing': 0.3,
    'legend.borderaxespad': 0.3,
})

# ---------------------------------------------------------------- shared labels and colours
YLAB = r'$[H_*R_*\Omega_{\rm vac}]^{-2}\,d\Omega_{\rm GW}/d\ln k$'
YLAB_RATIO_SPEC = r'$\mathcal{P}^{\rm sur}_{\rm GW}/\mathcal{P}^{\rm lat}_{\rm GW}$'   # surrogate / lattice (lattice = reference)
YLAB_RATIO_IK = r'$I^{\rm sur}_{\mathcal{K}}/I^{\rm lat}_{\mathcal{K}}$'
YLAB_RATIO_SPEC_RED = r'$\mathcal{P}^{\rm red}_{\rm GW}/\mathcal{P}^{\rm lat}_{\rm GW}$'   # reduced calculation / lattice (N_b = 2)
YLAB_RATIO_IK_RED = r'$I^{\rm red}_{\mathcal{K}}/I^{\rm lat}_{\mathcal{K}}$'

COL3 = ['#0072B2', '#D55E00', '#009E73']          # the three pairs of each N_b=2/3 configuration
LB_LABEL = {0.84: '0.84', 0.069: '0.069', P.LB845: '0.845'}
WALL = {0.84: 'thin', 0.069: 'thick', P.LB845: 'thin'}   # "<wall> bubbles" in panel labels;
                                                        # NB: Cutting et al. call lambda_bar >= 0.84 thin-wall


def ticks(ax):
    ax.tick_params(axis='both', which='both', direction='in', top=True, right=True)


def panel_label(ax, lb, extra=''):
    """Bold wall type, lambda_bar and an optional suffix, as the panel title."""
    ax.set_title(rf'\textbf{{{WALL[lb]} wall}} ($\bar\lambda={LB_LABEL[lb]}$)' + extra)


def hide_join_label(ax, x=1.0):
    """Hide the tick label at x in a right-hand column (it would collide with the left column's last label)."""
    ax.figure.canvas.draw()
    for t in ax.xaxis.get_major_ticks():
        if abs(t.get_loc() - x) < 1e-9:
            t.label1.set_visible(False)

## $N_b=2$ spectra at $t/R_*\simeq1.095$ (lattice snapshot nearest $t_m$; surrogate at that time)

In [ ]:
n2 = P.cached('fig12_n2', P.build_n2, REBUILD)

# ---------------------------------------------------------------- settings
FIGSIZE = (TEXTWIDTH, 0.52 * TEXTWIDTH)
HEIGHT_RATIOS = [2.2, 1]

XLIM = (0.85, 150)
YLIM = {0.84: (4e-7, 4e-4),          # spectra, per column (each has its own y-axis)
        0.069: (2e-7, 2e-4)}
YLIM_RATIO = {0.84: (0.6, 1.4),      # ratio, per column
              0.069: (0.6, 1.4)}
YTICKS_RATIO = {0.84: [0.6, 0.8, 1.0, 1.2, 1.4],
                0.069: [0.6, 0.8, 1.0, 1.2, 1.4]}

#GAMMA_LABEL_X0 = np.array([[0.98, 0.98, 0.98], [0.46, 0.32, 0.28]])
#GAMMA_LABEL_Y0 = np.array([[0.98, 0.83, 0.67], [0.9, 0.765, 0.6]])
GAMMA_LABEL_X = 0.1                # gamma_coll labels: left, below the legend
GAMMA_LABEL_Y0 = 0.4
GAMMA_LABEL_DY = 0.075
LEGEND_LOC = 'lower left'

# ---------------------------------------------------------------- figure
fig, axes = plt.subplots(
    2, 2, figsize=FIGSIZE, sharex='col',
    gridspec_kw=dict(height_ratios=HEIGHT_RATIOS, hspace=0, wspace=0.15),
)

for col, lb in enumerate([0.84, 0.069]):
    a, b = axes[:, col]
    pairs = [(k, v) for k, v in n2.items() if k[0] == lb]

    for (key, r), cc in zip(pairs, COL3):
        # spectra: lattice with mode-counting band, surrogate dashed
        a.plot(r['kR_sh'], r['y_sh'], color=cc, lw=LW)
        a.fill_between(r['kR_sh'], np.maximum(r['y_sh'] - r['yerr'], 1e-300), r['y_sh'] + r['yerr'],
                       color=cc, alpha=0.2, lw=0)
        m = (r['kR_rec'] >= 1) & (r['kR_rec'] <= 100)
        a.plot(r['kR_rec'][m], r['y_rec'][m], color=cc, lw=LW_THIN, ls='--')

        # ratio through the lattice shell estimator
        x = r['kR_sh'][r['shell_mask']]
        y = r['rec_shells'] / r['y_sh'][r['shell_mask']]                       # surrogate / lattice
        e = y * r['yerr'][r['shell_mask']] / r['y_sh'][r['shell_mask']]        # lattice uncertainty, relative
        b.plot(x, y, color=cc, lw=LW_THIN)
        b.fill_between(x, y - e, y + e, color=cc, alpha=0.2, lw=0)

    # legend and labels
    a.plot([], [], color='black', lw=LW, label='3D lattice')
    a.plot([], [], color='black', lw=LW_THIN, ls='--', label='reduced calculation')
    a.legend(frameon=False, loc=LEGEND_LOC)
    panel_label(a, lb, r', $N_b=2$')
    for i, ((key, r), cc) in enumerate(zip(pairs, COL3)):
        a.text(GAMMA_LABEL_X, GAMMA_LABEL_Y0 - GAMMA_LABEL_DY * i, rf"$\gamma_{{\rm coll}}\simeq{round(r['gamma'])}$",
               color=cc, transform=a.transAxes, ha='left', va='top')

    # axes
    a.set_xscale('log')
    a.set_yscale('log')
    a.set_xlim(*XLIM)
    a.set_ylim(*YLIM[lb])
    
    b.axhline(1, color='gray', ls=':', lw=LW_THIN)
    b.set_ylim(*YLIM_RATIO[lb])
    b.set_yticks(YTICKS_RATIO[lb])
    b.set_xlabel(r'$kR_*$')

    for ax in (a, b):
        ticks(ax)

axes[0, 0].set_ylabel(YLAB)
axes[1, 0].set_ylabel(YLAB_RATIO_SPEC_RED)

fig.savefig(OUT / 'n2_spectra.pdf', bbox_inches='tight')
plt.show()

for k, r in n2.items():
    print(k, f"gamma={r['gamma']:.3f}: lattice snapshot t/R_*={r['t_snap']/r['d']:.3f}")

## $N_b=2$ integrated power $I_{\mathcal K}(t)$ up to the first periodic-image contact

In [ ]:
import matplotlib.transforms
# ---------------------------------------------------------------- settings
FIGSIZE = (TEXTWIDTH, 0.52 * TEXTWIDTH)
HEIGHT_RATIOS = [2.2, 1]

T_FIG1 = 1.095                       # t/R_* of the N_b=2 spectra (vertical line)
XLIM = (0.5, None)
YLIM = {0.84: (-1e-5, 5), 0.069: (-1e-5, 1.8)}    # I_K, per column (different scales)
YLIM_RATIO = (0.95, 1.06)
YTICKS_RATIO = [0.96, 0.98, 1.0, 1.02, 1.04]

GAMMA_LABEL_X = 0.03                 # gamma_coll labels: left, below the legend
GAMMA_LABEL_Y0 = 0.3
GAMMA_LABEL_DY = 0.075
LEGEND_LOC = 'upper left'
LEGEND_ANCHOR = (0.37, 0.99)          # legend's upper-left corner (axes fraction): top middle, right of the t/d_ij line
UNIT = 1e-3                          # I_K plotted in units of UNIT

# ---------------------------------------------------------------- figure
fig, axes = plt.subplots(
    2, 2, figsize=FIGSIZE, sharex='col', sharey='row' if False else False,
    gridspec_kw=dict(height_ratios=HEIGHT_RATIOS, hspace=0, wspace=0.15),
)

for col, lb in enumerate([0.84, 0.069]):
    a, b = axes[:, col]
    pairs = [(k, v) for k, v in n2.items() if k[0] == lb]

    for (key, r), cc in zip(pairs, COL3):
        x = r['t'] / r['d']
        a.plot(x, r['I_lat'] / UNIT, color=cc, lw=LW)
        a.plot(x, r['I_red'] / UNIT, color=cc, lw=LW_THIN, ls='--')
        ok = (r['I_lat'] > 1e-3 * r['I_lat'].max()) & (r['I_red'] > 1e-3 * r['I_lat'].max())
        b.plot(x[ok], (r['I_red'] / r['I_lat'])[ok], color=cc, lw=1.1)

    # legend and labels
    a.plot([], [], color='black', lw=LW, label='3D lattice')
    a.plot([], [], color='black', lw=LW_THIN, ls='--', label='reduced calculation')
    a.legend(frameon=False, loc=LEGEND_LOC, bbox_to_anchor=LEGEND_ANCHOR)
    panel_label(a, lb, r', $N_b=2$')
    for i, ((key, r), cc) in enumerate(zip(pairs, COL3)):
        a.text(GAMMA_LABEL_X, GAMMA_LABEL_Y0 - GAMMA_LABEL_DY * i, rf"$\gamma_{{\rm coll}}\simeq{round(r['gamma'])}$",
               color=cc, transform=a.transAxes, ha='left', va='top')

    a.text(T_FIG1 - 0.03, 0.88, r"$\frac{t}{d_{ij}}$", color='gray', ha='right',   # left of the t/d_ij line
           transform=matplotlib.transforms.blended_transform_factory(a.transData, a.transAxes))
    
    # axes
    for ax in (a, b):
        ax.axvline(T_FIG1, color='gray', ls='--', lw=LW_THIN)
        ticks(ax)
    a.set_xlim(*XLIM)
    a.set_ylim(*YLIM[lb])
    a.set_yticks([v for v in a.get_yticks() if 0 < v <= YLIM[lb][1]])   # no 0 at the join, nothing above ymax (set_yticks would extend the axis)
    #a.set_yscale('log')

    b.axhline(1, color='gray', ls=':', lw=LW_THIN)
    b.set_ylim(*YLIM_RATIO)
    b.set_yticks(YTICKS_RATIO)
    b.set_xlabel(r'$t/R_*$')

axes[0, 0].set_ylabel(r'$I_{\mathcal{K}}(t)\ [10^{-3}]$')
axes[1, 0].set_ylabel(YLAB_RATIO_IK_RED)

fig.savefig(OUT / 'n2_integrated_power.pdf', bbox_inches='tight')
plt.show()

## $N_b=3$ spectra (lattice snapshot nearest the latest pair $t_m$; damped surrogate at that time)

In [ ]:
n3 = P.cached('fig3_n3', P.build_n3, REBUILD)

# ---------------------------------------------------------------- settings
FIGSIZE = (TEXTWIDTH, 0.52 * TEXTWIDTH)
HEIGHT_RATIOS = [2.2, 1]
COLOR = 'black'                      # one curve per panel

XLIM = (0.85, 150)
YLIM = {0.84: (8e-6, 1e-3),          # spectra, per column (each has its own y-axis)
        0.069: (1.3e-6, 5e-4)}
YLIM_RATIO = {0.84: (0.6, 1.7),      # ratio, per column
              0.069: (0.6, 1.7)}
YTICKS_RATIO = {0.84: [0.6, 0.8, 1.0, 1.2, 1.4, 1.6],
                0.069: [0.6, 0.8, 1.0, 1.2, 1.4, 1.6]}
LEGEND_LOC = 'lower left'

# ---------------------------------------------------------------- figure
fig, axes = plt.subplots(
    2, 2, figsize=FIGSIZE, sharex='col',
    gridspec_kw=dict(height_ratios=HEIGHT_RATIOS, hspace=0, wspace=0.15),
)

for col, lb in enumerate([0.84, 0.069]):
    a, b = axes[:, col]
    r = n3[lb]

    a.plot(r['kR_sh'], r['y_sh'], color=COLOR, lw=LW, label='3D lattice')
    a.fill_between(r['kR_sh'], np.maximum(r['y_sh'] - r['yerr'], 1e-300), r['y_sh'] + r['yerr'],
                   color=COLOR, alpha=0.2, lw=0)
    m = (r['kR_rec'] >= 1) & (r['kR_rec'] <= 100)
    a.plot(r['kR_rec'][m], r['y_rec'][m], color=COLOR, lw=LW_THIN, ls='--', label='surrogate')
    #a.plot(r['kR_rec'][m], r['y_rec_raw'][m], color='red', lw=LW_THIN, ls=LS_UNDAMPED, label='surrogate, undamped')

    x = r['kR_sh'][r['shell_mask']]
    y = r['rec_shells'] / r['y_sh'][r['shell_mask']]               # surrogate / lattice
    e = y * r['yerr'][r['shell_mask']] / r['y_sh'][r['shell_mask']]    # lattice uncertainty, relative
    b.plot(x, y, color=COLOR, lw=LW_THIN)
    #b.plot(x, r['rec_shells_raw'] / r['y_sh'][r['shell_mask']], color='red', lw=LW_THIN, ls=LS_UNDAMPED)
    b.fill_between(x, y - e, y + e, color=COLOR, alpha=0.2, lw=0)

    a.legend(frameon=False, loc=LEGEND_LOC)
    panel_label(a, lb, rf", $N_b=3$")
    a.text(0.96, 0.95, rf"$t/R_*={r['t_snap']/r['R']:.2f}$", transform=a.transAxes, ha='right', va='top')

    a.set_xscale('log')
    a.set_yscale('log')
    a.set_xlim(*XLIM)
    a.set_ylim(*YLIM[lb])

    b.axhline(1, color='gray', ls=':', lw=LW_THIN)
    b.set_ylim(*YLIM_RATIO[lb])
    b.set_yticks(YTICKS_RATIO[lb])
    b.set_xlabel(r'$kR_*$')

    for ax in (a, b):
        ticks(ax)

axes[0, 0].set_ylabel(YLAB)
axes[1, 0].set_ylabel(YLAB_RATIO_SPEC)

fig.savefig(OUT / 'n3_spectra.pdf', bbox_inches='tight')
plt.show()

## $N_b>3$: late-time spectra and integrated power, with ratios ($\gamma_*=4,5,6$; $\bar\lambda=0.845$ and $0.069$)

In [ ]:
LATTICE_CENTRAL = 'last'             # lattice central value: 'last' snapshot or 'median' over the late-time window (separate caches)
nm = P.cached('fig45_nmany' + ('_last' if LATTICE_CENTRAL == 'last' else ''), lambda: P.build_nmany(LATTICE_CENTRAL), REBUILD)


def lattice_grid(run):
    """Grid points per side of a sledgehamr run, from its first spectrum file's header."""
    f = sorted((P.SH / run / 'gw_spectra').glob('*/spectra.hdf5'))[0]
    with P.h5py.File(f) as h:
        return int(h['Header'][1])


# ================================================================ spectra
# ---------------------------------------------------------------- settings
GRID = 2048                          # only lattice runs on this grid are shown
FIGSIZE = (TEXTWIDTH, 0.85 * TEXTWIDTH)   # gives the same panel aspect ratio as the N_b=2 and 3 spectra
HEIGHT_RATIOS = [2.2, 1]
BLOCK_GAP = 0.0                      # extra vertical gap between the thick- and thin-wall blocks (fraction)
BLOCK_TOP = 0.918                    # top of the panels within each block (room for the block title)
BLOCK_BOTTOM = 0.153                 # bottom of the panels within each block (room for the x labels)
COLS = [4, 5, 6]                     # gamma_* per column
BLOCKS = [P.LB845, 0.069]            # lambda_bar of the top and bottom block

XLIM = (0.8, 160)
YLIM = {P.LB845: (8e-5, 1e-2),       # spectra, per block
        0.069: (4e-5, 1e-2)}
YLIM_RATIO = {P.LB845: (0.2, 1.6),
              0.069: (0.2, 1.6)}
YTICKS_RATIO = [0.4, 0.7, 1.0, 1.3]
LEGEND_LOC = 'lower left'            # legend in the left panel of each block
SHOW_CUTTING = False                 # overlay Cutting et al. 2020 and our N_b=512 surrogate in the gamma_*=4 panels
CUTTING = {lb: np.loadtxt(P.DATA / f'cutting2020/fig10d_tR4.0_lb{tag}_vector.txt', delimiter=',')   # Cutting et al. 2020,
           for lb, tag in ((P.LB845, '0.845'), (0.069, '0.069'))}                            # Fig. 10(d): gamma_*=4, N_b=512, t/R_*=4
CUTTING_STYLE = dict(color='#d62728', alpha=0.45, lw=LW_THIN)   # faint red, drawn underneath our curves
CUT_SURR = {lb: P.cutting_surrogate(lb) for lb in (P.LB845, 0.069)}   # our surrogate for their setup: 16 random N_b=512 placements
CUT_SURR_STYLE = dict(color='#d62728', lw=LW_THIN, ls='--')
CUT_SURR_ALPHA = 0.15
GAMMA_LABEL_POS = [(0.04, 0.95), (0.96, 0.95)]       # gamma_* label, axes fraction (top right)

# ---------------------------------------------------------------- data selection
SHOW = {key: r for key, r in nm.items() if lattice_grid(r['run']['sh']) == GRID}


def run_for(lb, gp):
    runs = [r for r in SHOW.values() if r['run']['lb'] == lb and r['run']['panel'] == gp]
    assert len(runs) == 1, (lb, gp, len(runs))
    return runs[0]


# ---------------------------------------------------------------- figure
fig = plt.figure(figsize=FIGSIZE)
blocks = fig.subfigures(2, 1, hspace=BLOCK_GAP)

for ib, (sub, lb) in enumerate(zip(blocks, BLOCKS)):
    axes = sub.subplots(2, 3, sharex=True, sharey='row',
                        gridspec_kw=dict(height_ratios=HEIGHT_RATIOS, wspace=0, hspace=0))
    sub.subplots_adjust(top=BLOCK_TOP, bottom=BLOCK_BOTTOM)
    axes[0, 1].set_title(rf'\textbf{{{WALL[lb]} wall}} ($\bar\lambda={LB_LABEL[lb]}$), $N_b\gg1$')

    for c, gp in enumerate(COLS):
        a, b = axes[0, c], axes[1, c]
        r = run_for(lb, gp)

        a.fill_between(r['kR_sh'], r['y_lo'], r['y_hi'], color='black', alpha=0.18, lw=0)
        a.plot(r['kR_sh'], r['y_med'], color='black', lw=LW, label='3D lattice (this work)' if SHOW_CUTTING else '3D lattice')
        if SHOW_CUTTING and gp == 4:                       # Cutting et al. 2020 ran gamma_* = 4 only
            a.plot(*CUTTING[lb].T, zorder=1, label='3D lattice (Cutting et al.)', **CUTTING_STYLE)
            cs = CUT_SURR[lb]; mk = (cs['kR'] >= 1) & (cs['kR'] <= 100)
            a.fill_between(cs['kR'][mk], cs['ens'][:, mk].min(axis=0), cs['ens'][:, mk].max(axis=0), color=CUT_SURR_STYLE['color'], alpha=CUT_SURR_ALPHA, lw=0, zorder=1)
            a.plot(cs['kR'][mk], np.median(cs['ens'][:, mk], axis=0), zorder=1, label=r'surrogate ($N_b=512$)', **CUT_SURR_STYLE)
        m = (r['kR_rec'] >= 1) & (r['kR_rec'] <= 100)
        a.plot(r['kR_rec'][m], r['y_rec'][m], color='black', lw=LW_THIN, ls='--', label='surrogate')
        a.plot(r['kR_rec'][m], r['y_rec_raw'][m], color='black', lw=LW_THIN, ls=LS_UNDAMPED, label='surrogate, undamped')

        x = r['kR_sh'][r['shell_mask']]
        s = r['rec_shells']
        b.fill_between(x, s / r['y_hi'][r['shell_mask']], s / r['y_lo'][r['shell_mask']],      # surrogate / lattice
                       color='black', alpha=0.18, lw=0)
        b.plot(x, s / r['y_med'][r['shell_mask']], color='black', lw=LW_THIN)
        #b.plot(x, r['rec_shells_raw'] / r['y_med'][r['shell_mask']], color='black', lw=LW_THIN, ls=LS_UNDAMPED)

        if c == 0:
            a.legend(frameon=False, loc=LEGEND_LOC)
        a.text(*GAMMA_LABEL_POS[col], rf'$\gamma_*\simeq{gp}$', transform=a.transAxes, ha='right', va='top')

        a.set_xscale('log')
        a.set_yscale('log')
        a.set_xlim(*XLIM)
        a.set_ylim(*YLIM[lb])

        b.axhline(1, color='gray', ls=':', lw=LW_THIN)
        b.set_ylim(*YLIM_RATIO[lb])
        b.set_yticks(YTICKS_RATIO)
        b.set_xlabel(r'$kR_*$')

        for ax in (a, b):
            ticks(ax)

    axes[0, 0].set_ylabel(YLAB)
    axes[1, 0].set_ylabel(YLAB_RATIO_SPEC)
    for c in (1, 2):
        hide_join_label(axes[1, c])

fig.savefig(OUT / 'nmany_spectra.pdf', bbox_inches='tight')
plt.show()


# ================================================================ integrated power, N_b >> 1

# ---------------------------------------------------------------- settings
FIGSIZE = (TEXTWIDTH, 0.52 * TEXTWIDTH)
HEIGHT_RATIOS = [2.2, 1]
PANELS = [P.LB845, 0.069]            # lambda_bar of the left and right column (same order as the other figures)

XLIM = [(0, 4.5), (0.,8.5)]
XTICKS = [0, 2, 4, 6, 8]
UNIT = 1e-3                          # I_K plotted in units of UNIT
YLIM = {0.069: (0, 12.2),              # I_K, per column
        P.LB845: (0, 12.2)}
YLIM_RATIO = (0.2, 1.4)
YTICKS_RATIO = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2]

GAMMA_LABEL_X = [0.04, 0.76]                 # gamma_* labels (axes fraction), top right
GAMMA_LABEL_Y0 = 0.95
GAMMA_LABEL_DY = 0.075
LEGEND_LOC = 'lower right'

# ---------------------------------------------------------------- figure
fig, axes = plt.subplots(
    2, 2, figsize=FIGSIZE, sharex='col',
    gridspec_kw=dict(height_ratios=HEIGHT_RATIOS, hspace=0, wspace=0.15),
)

for col, lb in enumerate(PANELS):
    a, b = axes[:, col]

    for gp, cc in zip(COLS, COL3):
        r = run_for(lb, gp)
        x = r['t'] / r['R']
        a.plot(x, r['I_lat'] / UNIT, color=cc, lw=LW)
        a.plot(x, r['I_red'] / UNIT, color=cc, lw=LW_THIN, ls='--')
        a.plot(x, r['I_red_raw'] / UNIT, color=cc, lw=LW_THIN, ls=LS_UNDAMPED)
        ok = (r['I_lat'] > 1e-3 * r['I_lat'].max()) & (r['I_red'] > 1e-3 * r['I_lat'].max())
        b.plot(x[ok], (r['I_red'] / r['I_lat'])[ok], color=cc, lw=LW_THIN)
        #b.plot(x[ok], (r['I_red_raw'] / r['I_lat'])[ok], color=cc, lw=LW_THIN, ls=LS_UNDAMPED)

    # legend and labels
    a.plot([], [], color='black', lw=LW, label='3D lattice')
    a.plot([], [], color='black', lw=LW_THIN, ls='--', label='surrogate')
    a.plot([], [], color='black', lw=LW_THIN, ls=LS_UNDAMPED, label='surrogate, undamped')
    a.legend(frameon=False, loc=LEGEND_LOC)
    panel_label(a, lb, r', $N_b\gg1$')
    for i, (gp, cc) in enumerate(zip(COLS, COL3)):
        a.text(GAMMA_LABEL_X[col], GAMMA_LABEL_Y0 - GAMMA_LABEL_DY * i, rf'$\gamma_*\simeq{gp}$',
               color=cc, transform=a.transAxes, ha='left', va='top')

    # axes
    for ax in (a, b):
        ticks(ax)
    a.set_xlim(*XLIM[col])
    #a.set_xticks(XTICKS)
    a.set_ylim(*YLIM[lb])
    #a.set_yticks(a.get_yticks()[1:])             # drop the 0 label at the join with the ratio panel

    b.axhline(1, color='gray', ls=':', lw=LW_THIN)
    b.set_ylim(*YLIM_RATIO)
    b.set_yticks(YTICKS_RATIO)
    b.set_xlabel(r'$t/R_*$')

axes[0, 0].set_ylabel(r'$I_{\mathcal{K}}(t)\ [10^{-3}]$')
axes[1, 0].set_ylabel(YLAB_RATIO_IK)

fig.savefig(OUT / 'nmany_integrated_power.pdf', bbox_inches='tight')
plt.show()

## Comparison with Cutting et al. (2020)

Their Fig. 10(d) lattice spectra ($\gamma_*=4$, $N_b=512$, $t/R_*=4$; extracted from the PDF vector graphics) against our surrogate for the same setup: 16 random placements of 512 bubbles, damped weights, evaluated at $t/R_*=4$ (median and min–max band).

In [ ]:
# ---------------------------------------------------------------- settings
FIGSIZE = (TEXTWIDTH, 0.52 * TEXTWIDTH)
HEIGHT_RATIOS = [2.2, 1]
PANELS = [P.LB845, 0.069]            # lambda_bar of the left and right column (same order as the other figures)

XLIM = (0.85, 150)
YLIM = {P.LB845: (8e-5, 6e-3),       # spectra, per column (each has its own y-axis)
        0.069: (8e-5, 6e-3)}
YLIM_RATIO = {P.LB845: (0.6, 2.0),   # ratio, per column
              0.069: (0.6, 2.0)}
YTICKS_RATIO = {P.LB845: [0.2,0.6,1.0,1.4,1.8],
                0.069: [0.2,0.6,1.0,1.4,1.8]}
LAT_BAND_ALPHA = 0.2                 # their band: +-|P_high - P_medium| (high vs medium resolution)
SURR_HATCH = dict(facecolor='dimgrey', edgecolor='0.55', lw=0)   # our band: min-max over 16 realizations
LEGEND_LOC = 'lower left'

# ---------------------------------------------------------------- data
# Cutting et al. 2020, Fig. 10(d) (gamma_*=4, N_b=512, t/R_*=4), from the PDF vector paths: kR_*, P, band low, band high
cut_lat = {lb: np.loadtxt(P.DATA / f'cutting2020/fig10d_tR4.0_lb{tag}_vector_band.txt', delimiter=',')
           for lb, tag in ((P.LB845, '0.845'), (0.069, '0.069'))}
cut_surr = {lb: P.cutting_surrogate(lb) for lb in PANELS}   # our surrogate, 16 random placements of 512 bubbles, t = 4 R_*


def loglog(x, X, Y):
    return np.exp(np.interp(np.log(x), np.log(X), np.log(Y), left=np.nan, right=np.nan))


# ---------------------------------------------------------------- figure
fig, axes = plt.subplots(
    2, 2, figsize=FIGSIZE, sharex='col',
    gridspec_kw=dict(height_ratios=HEIGHT_RATIOS, hspace=0, wspace=0.15),
)

for col, lb in enumerate(PANELS):
    a, b = axes[:, col]
    k, y, lo, hi = cut_lat[lb].T
    cs = cut_surr[lb]; m = (cs['kR'] >= 1) & (cs['kR'] <= 100)
    s_med, s_lo, s_hi = np.median(cs['ens'], axis=0), cs['ens'].min(axis=0), cs['ens'].max(axis=0)

    # spectra
    a.plot(k, y, color='black', lw=LW, label='3D lattice (Cutting et al.)')
    a.fill_between(k, lo, hi, color='black', alpha=LAT_BAND_ALPHA, lw=0)
    #a.plot(cs['kR'][m], s_med[m], color='black', lw=LW_THIN, ls='--', label='surrogate')
    a.fill_between(cs['kR'][m], s_lo[m], s_hi[m], **SURR_HATCH, label='surrogate')

    # ratio at their wavenumbers (surrogate interpolated log-log)
    mk = (k >= cs['kR'][m][0]) & (k <= cs['kR'][m][-1])
    r = loglog(k[mk], cs['kR'], s_med) / y[mk]                                  # surrogate / lattice
    b.plot(k[mk], r, color='black', lw=LW_THIN)
    # combined 1 sigma: their resolution half-width (+-|P_high - P_medium|) and our realization scatter (standard
    # deviation over the 16 placements), added in quadrature as relative errors
    rel_lat = 0.5 * (hi[mk] - lo[mk]) / y[mk]
    ens_k = np.array([loglog(k[mk], cs['kR'], e) for e in cs['ens']])
    rel_sur = ens_k.std(axis=0, ddof=1) / loglog(k[mk], cs['kR'], s_med)
    rel = np.hypot(rel_lat, rel_sur)
    b.fill_between(k[mk], r * (1 - rel), r * (1 + rel), color='black', alpha=LAT_BAND_ALPHA, lw=0)

    panel_label(a, lb, r', $\gamma_*=4$, $N_b=512$')
    a.legend(frameon=False, loc=LEGEND_LOC)

    a.set_xscale('log')
    a.set_yscale('log')
    a.set_xlim(*XLIM)
    a.set_ylim(*YLIM[lb])

    b.axhline(1, color='gray', ls=':', lw=LW_THIN)
    b.set_ylim(*YLIM_RATIO[lb])
    b.set_yticks(YTICKS_RATIO[lb])
    b.set_xlabel(r'$kR_*$')

    for ax in (a, b):
        ticks(ax)

axes[0, 0].set_ylabel(YLAB)
axes[1, 0].set_ylabel(YLAB_RATIO_SPEC)

fig.savefig(OUT / 'literature_comparison.pdf', bbox_inches='tight')
plt.show()


## Predictions: $\gamma_*=16$, $N_b=512$ spectra; peak vs $\gamma_*$ ($N_b=128$); peak vs $N_b$ ($\gamma_*=16$). Solid: damped, dotted: undamped

In [ ]:
from ptbuilder.weight_scan_diagnostics import characteristic_scale, load_kinematics

gsf, nbf = P.load_families()

# ---------------------------------------------------------------- settings
FIGSIZE = (TEXTWIDTH, 0.75 * TEXTWIDTH)
HEIGHT_RATIOS = [1.5, 1]             # top: spectrum (full width); bottom: the two peak scans
WSPACE = 0.2
HSPACE = 0.35
TITLE_POS = (0.97, 0.94)            # panel titles, inside the axes (axes fraction, top right)
UNIT = 1e-3                          # peaks plotted in units of UNIT

LBS = P.fam_available()              # lambda_bar with a family cache (thin wall first)
CL = {P.LB845: '#0072B2', 0.069: '#D55E00'}
NBS = [4, 8, 16, 32, 64, 128, 256, 512]
GS = P.FAM_GS
MS = 2.5                             # marker size
# Cutting et al. 2020, Fig. 10(d): maximum of their spectrum at t/R_* = 4 (N_b=512, gamma_*=4), with their shaded band
# (difference between high- and medium-resolution runs) at that kR_*; extracted from the PDF vector graphics
CUTTING_PEAK = {P.LB845: (4.409e-3, 3.673e-3, 5.146e-3), 0.069: (3.840e-3, 3.151e-3, 4.529e-3)}   # value, low, high
CUTTING_DX = {P.LB845: -0.08, 0.069: 0.08}   # tiny horizontal offset so the two error bars don't overlap
CUTTING_ALPHA = 0.6                  # Cutting et al. points on top of ours, semi-transparent with hollow squares
CUTTING_MARKER = dict(marker='s', ms=3.5, mfc='none', ls='none', elinewidth=LW_THIN, capsize=1.5)
ENV_ALPHA = 0.35                     # opacity of the realization envelope in the spectrum panel
ENVELOPE = np.loadtxt(P.DATA / 'envelope_gstar16_overlay.txt', delimiter=',')   # envelope approximation (kR_*, P_GW)
ENVELOPE_STYLE = dict(color='black', lw=LW, ls=(0, (3, 1.4, 1, 1.4)))
GUIDES = [   # power-law guides: (kR_* start, value at start, kR_* end, exponent, label (kR_*, value), label above/below)
    (1.05, 0.95e-3, 1.40, 2, (1.13, 1.33e-3), 'bottom'),     # label above the line
    (1.10, 2.50e-4, 1.65, 3, (1.32, 3.55e-4), 'top'),        # label below the line
    (28.0, 1.00e-3, 70.0, -1, (45.7, 7.3e-4), 'bottom'),
    (28.0, 1.22e-4, 75.0, -2, (22.2, 7.1e-5), 'bottom'),
]
GUIDE_STYLE = dict(color='0.4', lw=0.8, ls=(0, (0.8, 1.32)))

YLIM_SPEC = (1e-5, None)
YLIM_PEAK = (None, None)

# ---------------------------------------------------------------- data
X = P.X_PLOT
R_NB = {lb: P.fam_rstar(lb, P.NB_GS) for lb in LBS}      # R_* of the N_b family (gamma_* = P.NB_GS)


def chw(dE, R, lb, L):
    return P.to_chw(dE, R, lb, L**3)

# ---------------------------------------------------------------- figure
fig = plt.figure(figsize=FIGSIZE)
grid = fig.add_gridspec(2, 2, height_ratios=HEIGHT_RATIOS, wspace=WSPACE, hspace=HSPACE)
a0 = fig.add_subplot(grid[0, :])
a1, a2 = fig.add_subplot(grid[1, 0]), fig.add_subplot(grid[1, 1])

for lb in LBS:
    L512 = nbf[(lb, 512, 0)][0]
    for which, ls in (('damped', '-'), ('undamped', ':')):
        if which == 'undamped':
            continue
        label = rf'surrogate: {WALL[lb]} wall ($\bar\lambda={LB_LABEL[lb]}$)' if which == 'damped' else None
        # top: gamma_* = P.NB_GS, N_b = 512 spectrum (median of 16)
        y = chw(np.median([nbf[(lb, 512, rz)][1][which] for rz in range(16)], axis=0), R_NB[lb], lb, L512)
        a0.plot(X, y, color=CL[lb], ls=ls, lw=LW, label=label)
        ens = np.array([chw(nbf[(lb, 512, rz)][1][which], R_NB[lb], lb, nbf[(lb, 512, rz)][0]) for rz in range(16)])
        a0.fill_between(X, ens.min(axis=0), ens.max(axis=0), color=CL[lb], alpha=ENV_ALPHA, lw=0)   # envelope of the 16 realizations

        # middle: peak vs gamma_* (N_b = 128)
        pk = [chw(np.median(gsf[(lb, s)][2][which], axis=0), gsf[(lb, s)][0], lb, gsf[(lb, s)][1]).max() for s in GS]
        a1.plot(GS, np.array(pk) / UNIT, ls=ls, marker='o' if which == 'damped' else None, ms=MS, color=CL[lb], lw=LW_THIN, label=label, zorder=3)   # on top of the Cutting et al. points
        if which == 'damped':
            pr = np.array([[chw(e, gsf[(lb, s)][0], lb, gsf[(lb, s)][1]).max() for e in gsf[(lb, s)][2][which]]
                           for s in GS])
            a1.fill_between(GS, pr.min(axis=1) / UNIT, pr.max(axis=1) / UNIT, color=CL[lb], alpha=0.15, lw=0)

        # right: peak vs N_b (gamma_* = P.NB_GS)
        pk = [chw(np.median([nbf[(lb, nb, rz)][1][which] for rz in range(16)], axis=0), R_NB[lb], lb,
                  nbf[(lb, nb, 0)][0]).max() for nb in NBS]
        a2.plot(NBS, np.array(pk) / UNIT, ls=ls, marker='o' if which == 'damped' else None, ms=MS, color=CL[lb], lw=LW_THIN, label=label)
        if which == 'damped':
            pr = np.array([[chw(nbf[(lb, nb, rz)][1][which], R_NB[lb], lb, nbf[(lb, nb, rz)][0]).max()
                            for rz in range(16)] for nb in NBS])
            a2.fill_between(NBS, pr.min(axis=1) / UNIT, pr.max(axis=1) / UNIT, color=CL[lb], alpha=0.15, lw=0)

a0.plot(*ENVELOPE.T, label='envelope approximation', **ENVELOPE_STYLE)
for k0, y0, k1, n, (kl, yl), va in GUIDES:
    a0.plot([k0, k1], [y0, y0 * (k1 / k0) ** n], **GUIDE_STYLE)
    a0.text(kl, yl, rf'$k^{{{n}}}$', color=GUIDE_STYLE['color'], ha='left', va=va)
a0.set_xscale('log')
a0.set_yscale('log')
a0.set_ylim(*YLIM_SPEC)
a0.set_xlim(1,100)
a0.set_xlabel(r'$kR_*$')
a0.set_ylabel(YLAB)
a0.text(*TITLE_POS, rf'$\gamma_*={P.NB_GS},\ N_b=512$', transform=a0.transAxes, ha='right', va='top')
a0.legend(frameon=False, loc='lower left')

for lb in LBS:
    v, lo, hi = CUTTING_PEAK[lb]
    a1.errorbar(4 + CUTTING_DX[lb], v / UNIT, yerr=[[(v - lo) / UNIT], [(hi - v) / UNIT]], color=CL[lb],
                label='Cutting et al. (2020)' if lb == LBS[0] else None, alpha=CUTTING_ALPHA, zorder=4, **CUTTING_MARKER)
a1.legend(frameon=False, loc='upper right', bbox_to_anchor=(1.0, 0.88))   # just below the N_b label, clear of the curves
a1.set_ylim(*YLIM_PEAK)
a1.set_xlabel(r'$\gamma_*$')
a1.set_ylabel(r'peak amplitude $[10^{-3}]$')
a1.text(*TITLE_POS, r'$N_b=128$', transform=a1.transAxes, ha='right', va='top')

a2.set_xscale('log')
a2.set_xlabel(r'$N_b$')
a2.xaxis.set_major_locator(FixedLocator([4, 16, 64, 256]))
a2.set_xticklabels(['4', '16', '64', '256'])
a2.xaxis.set_minor_locator(NullLocator())
a2.set_ylabel(r'peak amplitude $[10^{-3}]$')
a2.text(*TITLE_POS, rf'$\gamma_*={P.NB_GS}$', transform=a2.transAxes, ha='right', va='top')

for ax in (a0, a1, a2):
    ticks(ax)

fig.savefig(OUT / 'predictions.pdf', bbox_inches='tight')
plt.show()

## Weight-weighted distributions vs $\gamma_*$ ($N_b=128$, 16 realizations): pair $\gamma_{ij}/\gamma_*$ and last collision time $t_{\max}/R_*$
Raw and $\hat D^{3.6}$-damped weights. For damped weights $t_{\max}$ = last time the pair's weight is $\ge1\%$ of its maximum (damped weights never return exactly to zero).

In [ ]:
wh = P.load_weight_hists(REBUILD)
pp = {(lb, gs): v for lb in P.fam_available() for gs, v in P.pair_power(lb).items()}   # per-pair damped GW power

# ---------------------------------------------------------------- settings
FIGSIZE = (TEXTWIDTH, 0.50 * TEXTWIDTH)
GS = P.GAMMA_STARS
X_EDGES = np.arange(min(GS) - 0.5, max(GS) + 1.5)
Y_EDGES = {'gamma': np.linspace(0, 3.5, 81), 'tmax': np.linspace(0, 2.7, 81)}
Y_LABEL = {'gamma': r'$\gamma_{ij}/\gamma_*$', 'tmax': r'$t_{\max}/R_*$'}
CNORM = LogNorm(1e-5, 0.3)
XTICKS = [4, 8, 12, 16]
WEIGHT_BY = 'power'                  # 'power': each pair's GW power (integrated over kR_* = 1-100);
                                     # 'peak': its power at the spectral peak; 'weight': its integrated collision weight
CBAR_LABEL = {'power': 'fraction of GW power', 'peak': 'fraction of peak GW power', 'weight': 'fraction of weight'}

# ---------------------------------------------------------------- figure (damped weights)
for kind in ('damp',):                   # damped weights only
    fig, axes = plt.subplots(2, 2, figsize=FIGSIZE, sharex=True, gridspec_kw=dict(hspace=0.12, wspace=0.35))

    for ir, lb in enumerate(P.fam_available()):
        for ic, q in enumerate(('gamma', 'tmax')):
            ye = Y_EDGES[q]
            H = np.zeros((len(ye) - 1, len(GS)))
            for ix, gs in enumerate(GS):
                d = wh[(lb, gs)]
                v = d['gamma'] / gs if q == 'gamma' else d[f'tmax_{kind}']
                if WEIGHT_BY == 'weight':
                    W = d[f'W_{kind}']
                else:
                    assert np.array_equal(pp[(lb, gs)]['gamma'], d['gamma'])     # same pairs, same order
                    W = pp[(lb, gs)]['P_int' if WEIGHT_BY == 'power' else 'P_peak']
                ok = np.isfinite(v)
                H[:, ix], _ = np.histogram(v[ok], bins=ye, weights=W[ok])
                H[:, ix] /= max(H[:, ix].sum(), 1e-300)

            ax = axes[ir, ic]
            pcm = ax.pcolormesh(X_EDGES, ye, np.where(H > 0, H, np.nan), norm=CNORM, cmap='Greys',
                                shading='flat', rasterized=True)
            ax.set_ylabel(Y_LABEL[q])
            ax.set_xticks(XTICKS)
            ax.text(0.04, 0.95, rf'$\bar\lambda={LB_LABEL[lb]}$', transform=ax.transAxes, va='top')
            ticks(ax)
            if ir == 1:
                ax.set_xlabel(r'$\gamma_*$')

    fig.colorbar(pcm, ax=axes, label=CBAR_LABEL[WEIGHT_BY], fraction=0.04, pad=0.02)
    fig.savefig(OUT / f'weight_distributions_{kind}.pdf', bbox_inches='tight')
    plt.show()

## Where the power comes from: $N_b=512$, $\gamma_*=16$ spectral density in $t/R_*$ and $\gamma_{ij}/\gamma_*$ (damped)

In [ ]:
dec = P.load_decomposition(REBUILD)

# ---------------------------------------------------------------- settings
FIGSIZE = (TEXTWIDTH, 0.55 * TEXTWIDTH)
YLIM_T = (0, 2.5)
YLIM_G = (0, 2.5)
VMIN = 1e-6                          # lower end of the colour scale (all panels); values below are drawn in the lowest colour
LEVELS = (0.383, 0.683, 0.954)
BAND_STYLE = {0.383: dict(color='white', lw=LW, ls='-'),
              0.683: dict(color='1', lw=LW_THIN, ls='-.'),
              0.954: dict(color='1', lw=LW_THIN, ls=':')}
BAND_LABEL = {0.383: r'38\%', 0.683: r'68\%', 0.954: r'95\%'}
LEGEND_COLOR = '0.15'                # band lines in the legend (white would vanish on the white background)
CBAR_LABEL = [r'$d\mathcal{P}_{\rm GW}/d(t/R_*)$',   # P_GW = [H_* R_* Omega_vac]^-2 dOmega_GW/dln k
              r'$d\mathcal{P}_{\rm GW}/d(\gamma_{ij}/\gamma_*)$']
PEAK_LINE = dict(color='white', ls='--', lw=LW_THIN)     # vertical line at the kR_* of the spectral peak
PEAK_DOT = dict(color='white', marker='o', ms=1.5, ls='none')   # maximum of the 2D distribution along that line
LABEL_POS = (0.96, 0.95)             # wall label, axes fraction (top right)


def bands(H, edges, levels=LEVELS):
    """Per kR_* column: equal-tailed ranges containing `level` of the power along the vertical axis."""
    yc = 0.5 * (edges[:-1] + edges[1:])
    mass = H * np.diff(edges)[:, None]
    tot = mass.sum(axis=0)
    out = {}
    for lev in levels:
        lo, hi = np.full(H.shape[1], np.nan), np.full(H.shape[1], np.nan)
        for k in range(H.shape[1]):
            if tot[k] <= 0:
                continue
            cum = np.r_[0., np.cumsum(mass[:, k]) / tot[k]]
            yy = np.r_[edges[0], yc]
            lo[k] = np.interp(0.5 * (1 - lev), cum, yy)
            hi[k] = np.interp(1 - 0.5 * (1 - lev), cum, yy)
        out[lev] = (lo, hi)
    return out


kR = P.X_PLOT
ke = np.sqrt(kR[:-1] * kR[1:])
ke = np.r_[kR[0]**2 / ke[0], ke, kR[-1]**2 / ke[-1]]

# ---------------------------------------------------------------- figure
fig, axes = plt.subplots(2, 2, figsize=FIGSIZE, sharex=True, gridspec_kw=dict(hspace=0.12, wspace=0.45))

for ir, lb in enumerate(P.fam_available()):
    d = dec[lb]
    V = d['L']**3
    n = d['n_real']
    te = d['t'] / d['R']
    Ht = P.to_chw(d['h_t'] / n, d['R'], lb, V) / np.diff(te)[:, None]
    ge = d['edges_g'] / d['gamma_star']
    Hg = P.to_chw(d['h_g'] / n, d['R'], lb, V) / np.diff(ge)[:, None]

    for ic, (H, e, lab, ylim) in enumerate(((Ht, te, r'$t/R_*$', YLIM_T),
                                           (Hg, ge, r'$\gamma_{ij}/\gamma_*$', YLIM_G))):
        ax = axes[ir, ic]
        vmax = H.max()
        pcm = ax.pcolormesh(ke, e, np.maximum(H, VMIN * 1e-3),
                            norm=LogNorm(VMIN, vmax), cmap='gist_stern_r', shading='auto', rasterized=True)
        ipk = int(np.argmax((H * np.diff(e)[:, None]).sum(axis=0)))      # peak of the spectrum (integral over the vertical axis)
        ax.axvline(kR[ipk], **PEAK_LINE)
        yc = 0.5 * (e[:-1] + e[1:])
        ax.plot(kR[ipk], yc[np.argmax(H[:, ipk])], **PEAK_DOT)
        for lev, (lo, hi) in bands(H, e).items():
            ax.plot(kR, lo, label=BAND_LABEL[lev], **BAND_STYLE[lev])
            ax.plot(kR, hi, **BAND_STYLE[lev])

        ax.set_xscale('log')
        ax.set_ylim(*ylim)
        ax.set_ylabel(lab)
        ax.text(*LABEL_POS, rf'\textbf{{{WALL[lb]} wall}} ($\bar\lambda={LB_LABEL[lb]}$)', transform=ax.transAxes, ha='right', va='top')
        ticks(ax)
        if ir == 1:
            ax.set_xlabel(r'$kR_*$')
        fig.colorbar(pcm, ax=ax, fraction=0.06, pad=0.02, label=CBAR_LABEL[ic])

from matplotlib.lines import Line2D
handles = [Line2D([], [], color=LEGEND_COLOR, lw=BAND_STYLE[lev]['lw'], ls=BAND_STYLE[lev]['ls']) for lev in LEVELS]
labels = [BAND_LABEL[lev] for lev in LEVELS]
fig.legend(handles, labels, frameon=False, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.02))

fig.savefig(OUT / 'power_origin.pdf', bbox_inches='tight', dpi=300)
plt.show()

## Appendix -- geometric collision weights $\mathcal W_{ij}(t)$

Top: undamped, bottom: $\hat D^{3.6}$-damped. Columns: the $N_b=3$ run ($\bar\lambda=0.84$), one curve per pair; the $N_b=15$, $\gamma_*=6$ run and one $N_b=512$, $\gamma_*=16$ realization ($\bar\lambda=0.845$), all colliding pairs including periodic images, coloured by $\gamma_{\rm coll}$.

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, PowerNorm, LinearSegmentedColormap
from matplotlib.collections import LineCollection

# ---------------------------------------------------------------- settings
FIGSIZE = (TEXTWIDTH, 0.6 * TEXTWIDTH)
WSPACE = 0                            # panels touch, shared axes
HSPACE = 0
XLIM = {'n3': (0, 1.6), 'many': (0, 1.6), 'fam': (0, 1.6)}   # t/R_*, all starting at 0; covers the damped weights (> 1e-3 until ~2.1 / 2.3 R_*)
XTICKS = {'n3': [0, 0.5, 1, 1.5], 'many': [0.5, 1, 1.5], 'fam': [0.5, 1, 1.5]}   # no 0 at the joins
ROW_LABEL_POS = (0.97, 0.05)          # 'undamped' / 'damped' in the left panels (axes fraction, bottom right)
CBAR_BOX = [0.86, 0.30, 0.05, 0.45]   # colour bar inside the panel (axes fraction: x, y, width, height)
YLIM = (0, 1.05)
MANY_RUN = 'Cutting_lb0.845_N15_L211.4'             # middle: N_b = 15, gamma_* = 6, lambda_bar = 0.845
FAM_CASE = (P.LB845, 16, 512, 0)                    # right: lambda_bar, gamma_*, N_b, realization (gamma_*=16 family)
CMAP = 'inferno'#'viridis'
CMAP_RANGE = (0.0, 1.0)              # part of the colour map used; < 1 cuts off the top
CNORM_POWER = 0.75                   # colour ~ gamma_coll^CNORM_POWER; < 1 moves the colour change to lower gamma_coll
VMAX_FACTOR = 1.                    # colour scale extends to VMAX_FACTOR * max(gamma_coll): the lines never reach the white top
CMAP_USED = LinearSegmentedColormap.from_list('cmap_used', plt.get_cmap(CMAP)(np.linspace(*CMAP_RANGE, 256)))
MANY_LW = {'many': 0.5, 'fam': 0.15}
GAMMA_LABEL_POS = (0.94, 0.95)                       # gamma_coll labels in the left panel (axes fraction, top right)
GAMMA_LABEL_DY = 0.09

# ---------------------------------------------------------------- data (cached: paper_data.weights_examples)
lbf, gsf_, nbf_, rzf = FAM_CASE
WEX = P.weights_examples(MANY_RUN, FAM_CASE)
t3, w3, g3, R3, t3d, w3d = (WEX['n3'][k] for k in ('t', 'w', 'g', 'R', 'td', 'wd'))
tm, wm, gm, Rm, tmd, wmd = (WEX['many'][k] for k in ('t', 'w', 'g', 'R', 'td', 'wd'))
tf, wf, gf, Rf, tfd, wfd = (WEX['fam'][k] for k in ('t', 'w', 'g', 'R', 'td', 'wd'))


def many_panel(ax, t, w, g, R, lw):
    # all pairs as one LineCollection, low gamma_coll drawn last (on top); returns the colour normalisation
    norm = PowerNorm(CNORM_POWER, g.min(), VMAX_FACTOR * g.max()); order = np.argsort(g)[::-1]
    segs = [np.column_stack([t / R, w[i]]) for i in order]
    ax.add_collection(LineCollection(segs, colors=CMAP_USED(norm(g[order])), linewidths=lw, rasterized=True))
    return norm


# ---------------------------------------------------------------- figure
fig, axs = plt.subplots(2, 3, figsize=FIGSIZE, sharex='col', sharey='row', gridspec_kw=dict(wspace=WSPACE, hspace=HSPACE))

for ax, (t, w) in zip(axs[:, 0], ((t3, w3), (t3d, w3d))):
    tp = np.r_[0., t]                                # start the curves at t = 0 (weights are zero before contact)
    for row, cc in zip(w, COL3):
        ax.plot(tp / R3, np.r_[0., row], color=cc, lw=LW)
for i, (g, cc) in enumerate(zip(g3, COL3)):
    axs[0, 0].text(GAMMA_LABEL_POS[0], GAMMA_LABEL_POS[1] - GAMMA_LABEL_DY * i, rf'$\gamma_{{\rm coll}}\simeq{round(g)}$',
                   color=cc, transform=axs[0, 0].transAxes, ha='right', va='top')
axs[0, 0].set_title(r'$\bar\lambda=0.84$, $N_b=3$')

for axc, raw, dmp, g, R, key, lab in ((axs[:, 1], (tm, wm), (tmd, wmd), gm, Rm, 'many', r', $\gamma_*=6$, $N_b=15$'),
                                      (axs[:, 2], (tf, wf), (tfd, wfd), gf, Rf, 'fam', rf', $\gamma_*={gsf_}$, $N_b={nbf_}$')):
    for ax, (t, w) in zip(axc, (raw, dmp)):
        norm = many_panel(ax, np.r_[0., t], np.column_stack([np.zeros(len(w)), w]), g, R, MANY_LW[key])
    axc[0].set_title(rf'$\bar\lambda={LB_LABEL[P.LB845]}$' + lab)
    cax = axc[0].inset_axes(CBAR_BOX)
    cb = fig.colorbar(ScalarMappable(norm=norm, cmap=CMAP_USED), cax=cax)
    cb.ax.set_title(r'$\gamma_{\rm coll}$', fontsize=FONTSIZE)
    cb.ax.set_ylim(g.min(), g.max())             # bar shows only the range present in the data
    cb.ax.yaxis.set_ticks_position('left')       # tick labels inside the panel (the next panel covers the right side)

for ax, lab in zip(axs[:, 0], (r'$\mathcal W_{ij}(t)$', r'$\mathcal W_{ij}^{\rm dec}(t)$')):
    #ax.text(*ROW_LABEL_POS, lab, transform=ax.transAxes, ha='right', va='bottom')
    ax.set_ylabel(lab )
for ax, key in zip(axs[-1], ('n3', 'many', 'fam')):
    ax.set_xlim(*XLIM[key])
    ax.set_xticks(XTICKS[key])
    ax.set_xlabel(r'$t/R_*$')
for ax in axs.flat:
    ax.set_ylim(*YLIM)
    ticks(ax)
axs[0, 0].set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])  # no 0 at the row join
axs[1, 0].set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])

fig.savefig(OUT / 'weights_examples.pdf', bbox_inches='tight', dpi=300)
plt.show()